# 05 - Optimisation

Objectif : tester plusieurs hyperparametres (learning rate, batch size, dropout) et garder le meilleur modele.

Idee simple : Grid Search manuel avec 2-3 valeurs, on compare val_accuracy.

Fonctions testees : `conception_modele_cnn`, `compiler_modele`, `entrainer_modele`, `evaluer_modele`

In [ ]:
import sys
from pathlib import Path
import itertools
import pickle
import tensorflow as tf
import matplotlib.pyplot as plt

PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_preprocessing import chargement_pretraitement_dataset, preparer_donnees
from src.model_architecture import conception_modele_cnn, compiler_modele
from src.train import entrainer_modele
from src.evaluate import evaluer_modele

print("Imports OK")

In [ ]:
# 1. Donnees (charge une seule fois)
DATA_PATH = PROJECT_ROOT / "data" / "raw"
X, y = chargement_pretraitement_dataset(str(DATA_PATH))
X_train, X_test, y_train, y_test, le, class_weights = preparer_donnees(X, y)
print(f"Train {X_train.shape} - Test {X_test.shape} - classes {list(le.classes_)}")

In [ ]:
# 2. Grid Search simple
# On teste 2 learning rates x 2 batch sizes = 4 combinaisons
# epochs=5 pour aller vite (mets 15-30 pour le vrai test)

learning_rates = [0.0001, 0.001]
batch_sizes = [16, 32]
epochs = 5  # augmente a 15-30 pour optimiser vraiment

results = []
best_acc = 0
best_config = None
best_model_path = PROJECT_ROOT / "models" / "best_optimized.keras"

for lr, bs in itertools.product(learning_rates, batch_sizes):
    print(f"\n{'='*50}")
    print(f"Test : lr={lr} | batch_size={bs} | epochs={epochs}")
    print(f"{'='*50}")
    
    model = conception_modele_cnn(input_shape=(224,224,3), nb_classes=len(le.classes_))
    model = compiler_modele(model, learning_rate=lr)
    history = entrainer_modele(model, X_train, y_train, X_test, y_test, class_weights, epochs=epochs, batch_size=bs)
    
    # Evalue
    loss, acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"-> val_accuracy : {acc*100:.2f}%")
    results.append({"lr": lr, "batch_size": bs, "accuracy": acc, "loss": loss})
    
    if acc > best_acc:
        best_acc = acc
        best_config = {"lr": lr, "batch_size": bs, "accuracy": acc}
        model.save(str(best_model_path))
        print(f"   Nouveau meilleur modele sauvegarde !")

print("\n=== Resultats ===")
for r in results:
    print(f"lr={r['lr']} | bs={r['batch_size']} | acc={r['accuracy']*100:.2f}%")
print(f"\nMeilleur : {best_config}")
print(f"Fichier : {best_model_path}")

In [ ]:
# 3. Comparaison visuelle
labels = [f"lr={r['lr']}\nbs={r['batch_size']}" for r in results]
accs = [r['accuracy']*100 for r in results]

plt.figure(figsize=(8, 4))
plt.bar(labels, accs)
plt.ylabel("Accuracy %")
plt.title("Comparaison des configurations")
plt.ylim(0, 100)
for i, v in enumerate(accs):
    plt.text(i, v+1, f"{v:.1f}%", ha='center')
plt.tight_layout()
plt.show()
print(f"Meilleure config : {best_config}")

### Conclusion
- Garde le modele `models/best_optimized.keras` si meilleur que `best_model.keras`
- Copie-le pour Streamlit :
```python
import shutil
shutil.copy("models/best_optimized.keras", "models/best_model.keras")
```
- Ensuite relance l'app : `docker compose up -d` -> http://localhost:8501